In [2]:
# ==============================================================
# PINN Weighting Schemes — Analysis (current folder structure)
# ==============================================================

import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# Path to your results folder
results_dir = "model/data_dt_pinn_ic"
assert os.path.exists(results_dir), f"Results folder not found: {results_dir}"

# List files
files = os.listdir(results_dir)
print("📁 Found files:", len(files))

📁 Found files: 6


In [3]:
# Identify unique scheme names (Static, Gradient, etc.)
schemes = set()
for f in files:
    if f.endswith(".pth"):
        parts = f.split("_")
        scheme = parts[-1].replace(".pth", "")
        schemes.add(scheme)

schemes = sorted(list(schemes))
print("🔍 Detected weighting schemes:", schemes)

🔍 Detected weighting schemes: []


In [4]:
def extract_losses(base, scheme):
    """Load MAE and RMSE arrays for a given scheme."""
    mae_path = os.path.join(results_dir, f"{base}_{scheme}_mae.npy")
    rmse_path = os.path.join(results_dir, f"{base}_{scheme}_rmse.npy")

    mae = np.load(mae_path) if os.path.exists(mae_path) else None
    rmse = np.load(rmse_path) if os.path.exists(rmse_path) else None
    return mae, rmse

# Find common base (without suffix)
bases = [f.replace("_Static.pth", "").replace("_Gradient.pth", "") for f in files if f.endswith(".pth")]
base_name = os.path.commonprefix(bases).rstrip("_")
print("📌 Base name pattern detected:", base_name)

📌 Base name pattern detected: 


In [5]:
results = {}
for scheme in schemes:
    mae, rmse = extract_losses(base_name, scheme)
    if mae is not None and rmse is not None:
        results[scheme] = {"mae": mae, "rmse": rmse}
        print(f"✅ Loaded {scheme}: MAE shape {mae.shape}, RMSE shape {rmse.shape}")
    else:
        print(f"⚠️ Missing arrays for {scheme}")

In [6]:
summary = {}
for scheme, data in results.items():
    mae, rmse = data["mae"], data["rmse"]
    summary[scheme] = {
        "mean_mae": np.mean(mae),
        "max_mae": np.max(mae),
        "mean_rmse": np.mean(rmse),
    }

print("\n📊 Summary metrics:")
for scheme, vals in summary.items():
    print(f"{scheme:10s} | Mean MAE: {vals['mean_mae']:.4e} | Max MAE: {vals['max_mae']:.4e} | Mean RMSE: {vals['mean_rmse']:.4e}")


📊 Summary metrics:


In [7]:
schemes = list(summary.keys())
mean_mae = [summary[s]["mean_mae"] for s in schemes]
mean_rmse = [summary[s]["mean_rmse"] for s in schemes]
max_mae = [summary[s]["max_mae"] for s in schemes]

plt.figure(figsize=(7,4))
plt.bar(schemes, mean_rmse, label="Mean RMSE", alpha=0.7)
plt.bar(schemes, mean_mae, label="Mean MAE", alpha=0.7)
plt.ylabel("Error")
plt.title("Validation Error Comparison (MAE vs RMSE)")
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.show()

In [8]:
# ==============================================================
# PINN Weighting Schemes – Local Debug (No File I/O)
# ==============================================================

import torch, copy, os
from omegaconf import OmegaConf
from src.nn.nn_dataset import DataSampler
from src.ode.sm_models_d import SynchronousMachineModels
from src.nn.nn_actions import NeuralNetworkActions

# -------------------------------------------------------------
# Device detection (MPS on Mac, CUDA on GPU, CPU fallback)
# -------------------------------------------------------------
def detect_device():
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print(f"✅ Using device: {device}")
    return device

device = detect_device()
torch.autograd.set_detect_anomaly(False)

# -------------------------------------------------------------
# Load and adapt configuration for quick debugging
# -------------------------------------------------------------
cfg = OmegaConf.load("src/conf/setup_dataset_nn.yaml")

cfg.nn.type = "DynamicNN"
cfg.nn.lr = 1e-3
cfg.nn.optimizer = "Adam"        # avoid LBFGS issues
cfg.nn.num_epochs = 30           # quick run
cfg.nn.early_stopping = False
cfg.nn.weighting.update_weights_freq = 5
cfg.dataset.perc_of_data_points = 0.05
cfg.dataset.perc_of_col_points = 0.05

dataset_path = "data/SM_AVR_GOV/dataset_set5_mixed.pkl"
assert os.path.exists(dataset_path), "Dataset missing!"

# -------------------------------------------------------------
# Quick training wrapper (no file saving)
# -------------------------------------------------------------
def train_weighting_scheme(cfg, dataset_path, scheme_name, epochs):
    print(f"\n{'='*60}\n🚀 Training scheme: {scheme_name}\n{'='*60}")
    cfg_local = copy.deepcopy(cfg)
    cfg_local.nn.weighting.update_weight_method = scheme_name
    cfg_local.nn.num_epochs = epochs

    # Initialize data/model
    ds = DataSampler(cfg_local, dataset_path=dataset_path)
    modelling_full = SynchronousMachineModels(cfg_local)
    network = NeuralNetworkActions(cfg_local, modelling_full, data_loader=ds)

    # Train (no saving, no wandb)
    network.pinn_train2(
        num_of_skip_data_points=1,
        num_of_skip_col_points=1,
        num_of_skip_val_points=1,
        wandb_run=None,
    )

    # Print simple metrics
    print(f"\n✅ {scheme_name} training done.")
    print(f"   Final total loss: {network.loss_total.item():.3e}")
    print(f"   Data loss: {network.loss_data.item():.3e}")
    print(f"   dt loss: {network.loss_dt.item():.3e}")
    print(f"   PINN loss: {network.loss_pinn.item():.3e}")
    print(f"   IC loss: {network.loss_pinn_ic.item():.3e}")
    print(f"   Current weights: {network.weighting_scheme.weights.detach().cpu().numpy()}\n")
    return network


# -------------------------------------------------------------
# Run locally for a few schemes
# -------------------------------------------------------------
schemes = [
    #"Static",
    "Gradient",
    #"ID",
    #"DN",
    #"WB"
    #"Sam"
]

for scheme in schemes:
    net = train_weighting_scheme(cfg, dataset_path, scheme, epochs=30)

✅ Using device: cpu

🚀 Training scheme: Gradient
SM_AVR_GOV data
Loading data from: data/SM_AVR_GOV/dataset_set5_mixed.pkl
Number of training samples:  80000 Number of validation samples:  10000 Number of testing samples:  10000
Number of different initial conditions for collocation points:  100
['theta', 'omega', 'E_d_dash', 'E_q_dash', 'R_F', 'V_r', 'E_fd', 'P_sv', 'P_m'] Variables
[[-2, 2], [-1, 1], [0], [1], [1], [1.105], [1.08], [0.7048], [0.7048]] Set of values for init conditions
[10, 10, 1, 1, 1, 1, 1, 1, 1] Iterations per value
Selected deep learning model:  DynamicNN
Number of labeled training data: 4000 Number of collocation points: 5000 Number of collocation points (IC): 100 Number of validation data: 10000
Weights initialized as:  [1, 0.001, 0.0001, 0.001]  are updated with scheme:  Gradient every 5 epochs
📄 Logging training to model/data_dt_pinn_ic/training_log_Gradient.csv
getting in training
grad norms are tensor([4.7063e+00, 1.3308e-03, 3.0720e-01, 2.3115e-03])
grad no

KeyboardInterrupt: 